In [ ]:
import math
from typing import Optional, Tuple
from smolagents import tool

@tool
def calculate_cargo_travel_time(
    origin_coords : Tuple[float, float],
    destination_coords : Tuple[float, float],
    cruising_speed_kmh : Optional[float] = 750.0
)-> float:
    """
    Calcluate the travel time for cargo plane between two points on Earth using great-circle distance.
    
    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)
    
    """

    def to_radians(degrees: float)-> float:
        return degrees*(math.pi/180)

    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    EARTH_RADIUS_KM = 6371.0

    dlon = lon2-lon1
    dlat = lat2-lat1

    a = (
        math.sin(dlat/2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2) ** 2
    )

    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c
    actual_distance = 1.1 * distance

    flight_time = (actual_distance/cruising_speed_kmh) + 1.0
    return round(flight_time, 2)

#print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

In [ ]:
import truststore
truststore.inject_into_ssl()
import os
from PIL import Image
from smolagents import CodeAgent, LiteLLMModel, GoogleSearchTool, VisitWebpageTool
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from langfuse import get_client

SmolagentsInstrumentor().instrument()
get_client()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_BASE_URL = os.getenv("LANGFUSE_BASE_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


model = LiteLLMModel(
    model_id="anthropic/claude-sonnet-5",
    api_key=ANTHROPIC_API_KEY
)

# agent = CodeAgent(
#     model=model,
#     tools=[VisitWebpageTool(),GoogleSearchTool(), ]
# )

# agent.run("Whats the live gold price in Anantapur, Andhra Pradesh Today?")

#print(ANTHROPIC_API_KEY is not None)

# """ print(ANTHROPIC_API_KEY)
# print(LANGFUSE_PUBLIC_KEY)
# print(LANGFUSE_SECRET_KEY)
# print(LANGFUSE_BASE_URL) """

In [ ]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [ ]:
import inspect

print(inspect.signature(GoogleSearchTool))

In [ ]:
agent = CodeAgent(
    model= model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    additional_authorized_imports=["pandas"],
    max_steps = 20
)

In [ ]:
result = agent.run(task)

In [ ]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.
{task}
""")

print(detailed_report)

In [ ]:
web_agent = CodeAgent(
    model=model,
    tools=[GoogleSearchTool(), VisitWebpageTool(), calculate_cargo_travel_time],
    name = "web_agent",
    description = "Browses the web to find information",
    verbosity_level = 0,
    max_steps = 10
)

In [ ]:
from smolagents.utils import encode_image_base64, make_image_url
from smolagents import OpenAIServerModel

def check_reasoning_and_plot(final_answer, agent_memory, **kwargs):
    multimodal_model = OpenAIServerModel(model_id="gpt-4o",max_tokens = 8096, api_key= OPENAI_API_KEY)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role" : "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_url",
                    "image_url": {"url" : make_image_url(encode_image_base64(image))}
                }
            ]
        }
    ]
    output = multimodal_model(messages).content
    print("feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True
manager_agent = CodeAgent(
        model=model,
        tools=[calculate_cargo_travel_time],
        managed_agents = [web_agent],
        additional_authorized_imports=["geopandas",
        "plotly",
        "plotly.express",
        "plotly.graph_objects",
        "plotly.io"
        "shapely",
        "json",
        "pandas",
        "numpy",],
        verbosity_level = 2,
        planning_interval=5,
        final_answer_checks = [check_reasoning_and_plot],
        max_steps = 15
)

In [ ]:
manager_agent.visualize()

In [ ]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")